# PoC: Agent สกัดข้อมูลจากเปเปอร์ vs ที่คนกรอกเอง

**คำถาม:** ค่าที่ agent สกัดจาก PDF ตรงกับที่คนกรอกใน `Raw Data` แค่ไหน?

```
PDF ──read_pdf_text──▶ prompt ──LLM──▶ JSON ──validate──▶ รายการอุปกรณ์
                                                              │ compare
Raw Data ──processed/normalize──▶ ground truth ───────────────┘──▶ ตาราง match/mismatch/missed/extra
```
ตอนนี้ใช้ `MockLLM` (ไม่เรียก API) เพื่อพิสูจน์ว่าท่อและตัวเทียบทำงาน — ตัวเลขที่ได้จาก mock **ไม่ใช่ผลของ model**

In [1]:
import sys, random
sys.path.insert(0, ".")
import pandas as pd
from extractor.ground_truth import load_truth
from extractor.llm import MockLLM, OpenRouterLLM
from extractor.agent import extract_paper
from extractor.compare import compare_paper, summarize
from extractor.schema import FIELD_NAMES

truth = load_truth()
print(truth.shape, truth.doi_raw.nunique(), "papers")

(134, 10) 45 papers


## 1) Mock: จำลองคำตอบ model โดยบิดค่าจริงบางส่วน
ใช้เช็คว่า compare จับ mismatch / missed ได้จริง (แทนที่ด้วย model จริงภายหลัง)

In [2]:
random.seed(0)
def fake_answer(g):
    devs = []
    for _, r in g.iterrows():
        d = {k: (None if pd.isna(r.get(k)) else r.get(k)) for k in FIELD_NAMES}
        d["label"] = r["device_id"]
        if d["pce"] is not None and random.random() < 0.15: d["pce"] += 0.8      # อ่านผิด
        if d["voc"] is not None and random.random() < 0.15: d["voc"] = None       # หาไม่เจอ
        devs.append(d)
    return {"devices": devs}

canned = {doi: fake_answer(g) for doi, g in truth.groupby("doi_raw")}
llm = MockLLM(canned)
# llm = OpenRouterLLM(model="...")   # <- สลับตอนมี API key

## 2) รัน agent ทุกเปเปอร์ แล้วเทียบ

In [3]:
details = []
for doi, g in truth.groupby("doi_raw"):
    text = ""  # จริง: read_pdf_text(pdf_path(doi)) — ต้องกรอก papers_manifest.csv ก่อน
    preds = extract_paper(llm, doi, text)
    details.append(compare_paper(g, preds).assign(doi_raw=doi))
detail = pd.concat(details, ignore_index=True)
summarize(detail)

verdict,match,mismatch,missed,extra,accuracy
field,,,,,
etl,134,0,0,0,1.000
ff,72,2,0,0,0.973
htl,134,0,0,0,1.000
htl_treatment,74,0,1,1,0.987
jsc,72,0,0,0,1.000
pce,69,10,0,0,0.873
perovskite,132,0,0,0,1.000
voc,64,0,10,0,0.865


## 3) ดูเคสที่ไม่ตรง

In [4]:
detail[detail.verdict.isin(["mismatch", "missed", "extra"])].head(20)

,device_id,label,field,truth,pred,verdict,doi_raw
136,10.1007_s10854-017-8094-9_Sample4,10.1007_s10854-017-8094-9_Sample4,pce,9.1,9.9,mismatch,10.1007_s10854-017-8094-9
225,10.1016_j.solmat.2025.113858_Sample1,10.1016_j.solmat.2025.113858_Sample3,voc,1.06,None,missed,10.1016_j.solmat.2025.113858
227,10.1016_j.solmat.2025.113858_Sample1,10.1016_j.solmat.2025.113858_Sample3,ff,79.3,81.1,mismatch,10.1016_j.solmat.2025.113858
229,10.1016_j.solmat.2025.113858_Sample1,10.1016_j.solmat.2025.113858_Sample3,htl_treatment,NaN,PMMA,extra,10.1016_j.solmat.2025.113858
240,10.1016_j.solmat.2025.113858_Sample3,10.1016_j.solmat.2025.113858_Sample1,pce,20.17,20.99,mismatch,10.1016_j.solmat.2025.113858
243,10.1016_j.solmat.2025.113858_Sample3,10.1016_j.solmat.2025.113858_Sample1,ff,81.1,79.3,mismatch,10.1016_j.solmat.2025.113858
245,10.1016_j.solmat.2025.113858_Sample3,10.1016_j.solmat.2025.113858_Sample1,htl_treatment,PMMA,None,missed,10.1016_j.solmat.2025.113858
352,10.1021_acsaem.4c02471_Sample4,10.1021_acsaem.4c02471_Sample4,pce,14.35,15.15,mismatch,10.1021_acsaem.4c02471
400,10.1021_acsami.4c00249_Sample1,10.1021_acsami.4c00249_Sample1,pce,17.85,18.65,mismatch,10.1021_acsami.4c00249
537,10.1039_d2tc04215k_Sample2,10.1039_d2tc04215k_Sample2,voc,0.94,None,missed,10.1039_d2tc04215k


## ถัดไป (TODO)
1. กรอก `papers_manifest.csv` (DOI → ชื่อ PDF) — ชื่อไฟล์ใน `original paper/` ไม่ใช่ DOI
2. `pip install pymupdf` แล้วเปิดใช้ `read_pdf_text`; ตัดสินใจเรื่อง SI และตาราง
3. ใส่ `OPENROUTER_API_KEY` แล้วเทียบหลายโมเดล (เก็บ cost/latency ด้วย)
4. ปรับตัวจับคู่อุปกรณ์ (ตอนนี้ใช้ระยะ PCE อย่างเดียว) และขยายฟิลด์ใน `schema.py`
5. ฟิลด์ที่คนกรอกผิดเอง: ดู `pipeline/overrides/` — ถ้า agent "ผิด" อาจเป็นคนกรอกผิดก็ได้ ควรตรวจกับต้นฉบับก่อนตัดสินว่า agent พลาด